In [2]:
import google.generativeai as genai
import os
import time

# --- Configuration ---
API_KEY_FILE = "apikey.txt"
# MODEL_NAME = 'gemini-pro'
MODEL_NAME = 'gemini-2.5-flash-preview-05-20' # 高度な文脈理解が期待できるモデル
# MODEL_NAME = 'gemini-1.5-flash-latest' # 速度とコストのバランスが良いモデル


def get_api_key(filepath=API_KEY_FILE):
    """指定されたファイルからAPIキーを読み込む"""
    try:
        with open(filepath, "r") as f:
            return f.read().strip()
    except FileNotFoundError:
        print(f"エラー: APIキーファイル '{filepath}' が見つかりません。")
        return None

def conduct_scripted_conversation(api_key, initial_question, scripted_follow_ups):
    """
    LLMと事前に定義されたスクリプトに基づいて対話を行う。
    """
    if not api_key:
        print("APIキーが設定されていません。")
        return

    genai.configure(api_key=api_key)
    try:
        model = genai.GenerativeModel(MODEL_NAME)
        # 新しいチャットセッションを開始
        chat = model.start_chat(history=[])  # 空の履歴で開始
    except Exception as e:
        print(f"モデルまたはチャットの初期化中にエラーが発生しました: {e}")
        return

    # 最初の質問
    print("--------------------------------------------------")
    print(f"あなた (最初の質問):\n{initial_question.strip()}")
    try:
        response = chat.send_message(initial_question)
        print(f"\nLLMの応答:\n{response.text.strip()}")
    except Exception as e:
        print(f"最初の質問の送信中にエラーが発生しました: {e}")
        return # 最初の質問で失敗したら、以降の処理は行わない

    # 事前に定義された追加質問を順番に実行
    for question in scripted_follow_ups:
        # APIのレート制限を考慮して少し待機
        # (モデルやプランによっては不要な場合もありますが、念のため)
        time.sleep(1.2) # 1秒より少し長めに
        print("--------------------------------------------------")
        print(f"あなた (追加の質問):\n{question.strip()}")
        try:
            response = chat.send_message(question)
            print(f"\nLLMの応答:\n{response.text.strip()}")
        except Exception as e:
            print(f"追加の質問 '{question}' の送信中にエラーが発生しました: {e}")
            # エラーが発生した場合、次の質問に進むか、ここで停止するかを選択できます。
            # この例ではエラーメッセージを表示して続行します。
            # return # ここで停止させる場合

    print("--------------------------------------------------")
    print("事前に定義されたすべての質問が完了しました。")
    # print("\n---最終的なチャット履歴---")
    # for message in chat.history:
    # print(f"役割: {message.role}, 内容: {message.parts[0].text.strip()}")
    # print("--------------------------")


if __name__ == "__main__":
    api_key = get_api_key()

    if api_key:
        tsubame_question = """
つばめちゃんは渋谷駅から東急東横線に乗り、自由が丘駅で乗り換えました。
東急大井町線の大井町方面の電車に乗り換えたとき、各駅停車に乗車すべきところ、
間違えて急行に乗車してしまったことに気付きました。
自由が丘の次の急行停車駅で降車し、反対方向の電車で一駅戻った駅が
つばめちゃんの目的地でした。目的地の駅の名前を答えてください。
"""

        # ここに事前にプログラムしておく追加の質問をリストとして定義します
        predefined_follow_up_questions = [
            "さらに、つばめちゃんが自由が丘駅で乗り換えたとき、先ほどとは反対方向の急行電車に間違って乗車してしまった場合を考えます。目的地の駅に向かうため、自由が丘の次の急行停車駅で降車した後、反対方向の各駅停車に乗車した場合、何駅先の駅で降りれば良いでしょうか？"
            #"その駅の主な特徴や周辺情報について、2つか3つ教えてください。",
            #"もしつばめちゃんが、自由が丘駅で大井町方面の各駅停車に正しく乗っていた場合、目的地は何番目の駅になりますか？（自由が丘駅を1番目と数えずに）"
        ]

        conduct_scripted_conversation(api_key, tsubame_question, predefined_follow_up_questions)
    else:
        print("APIキーがないため、プログラムを実行できません。")

--------------------------------------------------
あなた (最初の質問):
つばめちゃんは渋谷駅から東急東横線に乗り、自由が丘駅で乗り換えました。
東急大井町線の大井町方面の電車に乗り換えたとき、各駅停車に乗車すべきところ、
間違えて急行に乗車してしまったことに気付きました。
自由が丘の次の急行停車駅で降車し、反対方向の電車で一駅戻った駅が
つばめちゃんの目的地でした。目的地の駅の名前を答えてください。

LLMの応答:
つばめちゃんの行動を順に追っていきましょう。

1.  **自由が丘駅で乗り換え**: 東急大井町線の大井町方面に乗車。
2.  **間違い**: 各駅停車に乗るべきところ、急行に乗ってしまった。
3.  **急行の次の停車駅で降車**: 自由が丘駅から大井町方面へ向かう東急大井町線の急行は、次の停車駅が「**大岡山**」です。
4.  **反対方向へ一駅戻る**: 大岡山駅から、二子玉川・溝の口方面（自由が丘方面）へ一駅戻ります。大岡山駅の自由が丘方面の隣の駅は「**緑が丘**」です。

したがって、つばめちゃんの目的地の駅は **緑が丘** 駅です。
--------------------------------------------------
あなた (追加の質問):
さらに、つばめちゃんが自由が丘駅で乗り換えたとき、先ほどとは反対方向の急行電車に間違って乗車してしまった場合を考えます。目的地の駅に向かうため、自由が丘の次の急行停車駅で降車した後、反対方向の各駅停車に乗車した場合、何駅先の駅で降りれば良いでしょうか？

LLMの応答:
つばめちゃんの状況を整理しましょう。

1.  **自由が丘駅で乗り換え**: 東急大井町線の**反対方向**の急行電車（二子玉川・溝の口方面）に間違って乗車。
2.  **急行の次の停車駅で降車**: 自由が丘駅から二子玉川・溝の口方面へ向かう東急大井町線の急行は、次の停車駅が「**等々力（Todoroki）**」です。
    *   つばめちゃんは等々力駅で降車します。
3.  **目的地**: 最初の問題で判明した目的地は「**緑が丘（Midorigaoka）**」駅です。
4.  **等々力から目的地へ移動**: 等々力駅から、目的地である緑が